In [2]:
# ============================================================================
# notebook: notebooks/03_diagnostics_and_validity.ipynb  (v12 — A' two diagnostics)
# Project: "Incidental vs. Engineered Approval"
# Stage 3 (A'): the composite EngineeredScore ensemble is DROPPED. Under the
#   pre-registered axis verdict NonFragility is excluded, and the surviving
#   2-axis ensemble has NO convergent validity (V4 FAIL). The data support two
#   heterogeneous diagnostics, not one composite score:
#     Diagnostic 2 (Typicality / Density) : convergent with realized default (V4).
#     Diagnostic 1 (Stability)            : validated by group-gap reproducibility
#                                           and separation, NOT by default (it does
#                                           not converge with default — different
#                                           kind of validity).
#   This notebook:
#     C2. build the two diagnostic axes (no ensemble score).
#     C3. V2 axis independence (raw scale).
#     C4. V3 not-confidence — per diagnostic.
#     C5. V4 convergent validity — DIAGNOSTIC 2 ONLY (density -> default).
#     C6. V4' the dropped ensemble — documented FAIL (why the ensemble is dropped).
#     C7. Diagnostic 1 own validity — Stability does NOT converge with default;
#         validate it by group separation instead.
#     C8. Typicality paradox descriptive (borderline).
#     C9. persist per-diagnostic artifacts.
# Reads results/ ; writes results/. Run from notebooks/.
# ============================================================================


# ---------------------------------------------------------------------------
# CELL 1 — Paths, imports, load Stage-2 indices + Stage-1 cohort
# ---------------------------------------------------------------------------
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm

ROOT    = Path("..").resolve()
DATA    = ROOT / "data"
RESULTS = ROOT / "results"

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
AXIS_CORR_THRESHOLD = 0.70
CONF_CORR_MAX = 0.30

idx    = pd.read_parquet(RESULTS / "stage2_indices_approved.parquet")
cohort = pd.read_parquet(RESULTS / "stage1_cohort.parquet")
stage0 = pd.read_parquet(RESULTS / "stage0_labeled.parquet")

AUDIT = (["LIMIT_BAL"] + [f"BILL_AMT{i}" for i in range(1,7)]
         + [f"PAY_AMT{i}" for i in range(1,7)])

idx["DEFAULT"]       = stage0.loc[idx.index, "DEFAULT"].values
idx["P_VIP"]         = cohort.loc[idx.index, "P_VIP_stage1"].values
idx["GROUP"]         = cohort.loc[idx.index, "GROUP"].values
idx["CELL"]          = cohort.loc[idx.index, "CELL"].values
idx["is_borderline"] = cohort.loc[idx.index, "VIP_BORDERLINE_s1"].values
for c in AUDIT:
    idx[c] = stage0.loc[idx.index, c].values

print(f"Approved: {len(idx):,}  |  borderline: {int(idx['is_borderline'].sum()):,}")


# ---------------------------------------------------------------------------
# CELL 2 — Build the TWO diagnostic axes. NO composite EngineeredScore.
#   Diagnostic 1: Stability          (A_Stability = Stability percentile)
#   Diagnostic 2: Typicality/Density (density_pct; paradox is density->default +)
#   NonFragility and LowDensity are retained ONLY for the appendix / references.
# ---------------------------------------------------------------------------
idx["A_Stability"]  = idx["Stability_pct"]          # Diagnostic 1
idx["density_pct"]  = idx["Density_pct"]            # Diagnostic 2 (raw direction)
idx["A_LowDensity"] = 1.0 - idx["Density_pct"]      # appendix (inverse view)
idx["A_NonFrag"]    = idx["NonFragility_pct"]       # appendix (excluded axis)

# Deliberately NO ensemble score is constructed (A' decision, R-18).
B = idx[idx["is_borderline"] == 1].copy()
print("Two diagnostics built (NO composite ensemble).")
print("Borderline analysis set:", len(B), f"(defaults: {int(B['DEFAULT'].sum())})")
print("Diagnostic 1 (Stability) summary:")
print(B["A_Stability"].describe().round(3).to_string())
print("Diagnostic 2 (density_pct) summary:")
print(B["density_pct"].describe().round(3).to_string())


# ---------------------------------------------------------------------------
# CELL 3 — V2: axis independence between the two diagnostics (RAW scale).
# The two diagnostics must be independent to count as separate signals.
# LowDensity is a monotone transform of Density, so the raw check is
# Stability vs Density.
# ---------------------------------------------------------------------------
raw2 = B[["Density", "Stability"]].corr().abs()
r_stab_dens = raw2.loc["Stability", "Density"]
print("V2 — two-diagnostic RAW correlation (borderline):")
print(f"  |corr(Stability, Density)| = {r_stab_dens:.3f}  (threshold {AXIS_CORR_THRESHOLD})")
V2_PASS = r_stab_dens < AXIS_CORR_THRESHOLD
print(f"  V2 {'PASS' if V2_PASS else 'REVIEW'} (diagnostics are independent)")

# Transparency: raw correlation including the excluded NonFragility axis
ref = B[["Density", "Stability", "NonFragility"]].corr().abs()
print("\n[reference] RAW correlation incl. excluded NonFragility (appendix):")
print(ref.round(3).to_string())
print(f"  Density-NonFragility |r| = {ref.loc['Density','NonFragility']:.3f} "
      f"> {AXIS_CORR_THRESHOLD}  => NonFragility excluded (R-16)")


# ---------------------------------------------------------------------------
# CELL 4 — V3: each diagnostic is NOT predicted-probability repackaging.
# Check corr(axis, p(x)) and the pseudo-R2 gain of the axis over p(x) alone.
# ---------------------------------------------------------------------------
def logit(frame, cols):
    return sm.Logit(frame["DEFAULT"].values, sm.add_constant(frame[cols])).fit(disp=0)

print("V3 — not-confidence, per diagnostic:")
m_p = logit(B, ["P_VIP"])
V3_flags = {}
for name, col in [("Diag1 Stability", "A_Stability"),
                  ("Diag2 Density",   "density_pct")]:
    c_px = stats.pearsonr(B[col], B["P_VIP"])[0]
    m_ax = logit(B, [col, "P_VIP"])
    gain = m_ax.prsquared - m_p.prsquared
    ok = (abs(c_px) < CONF_CORR_MAX) and (gain > 0.003)
    V3_flags[name] = ok
    print(f"  {name:16} corr(axis,p(x))={c_px:+.3f}  "
          f"pseudoR2 gain={gain:+.4f}  p(x)_p={m_ax.pvalues['P_VIP']:.2e}  "
          f"{'PASS' if ok else 'REVIEW'}")
print("  NOTE: absolute pseudo-R2 is small; V3 only shows each axis is NOT p(x).")


# ---------------------------------------------------------------------------
# CELL 5 — V4: convergent validity — DIAGNOSTIC 2 ONLY (density -> default).
# This is the paradox: higher density (more typical) => MORE default (positive).
# Diagnostic 2 is the only axis with convergent (default) validity.
# ---------------------------------------------------------------------------
m_d2 = logit(B, ["density_pct", "P_VIP"])
coef2, p2 = m_d2.params["density_pct"], m_d2.pvalues["density_pct"]
rho2, prho2 = stats.spearmanr(B["density_pct"], B["DEFAULT"])
V4_D2_PASS = (p2 < 0.05) and (coef2 > 0)   # paradox sign is POSITIVE
print("V4 — convergent validity, DIAGNOSTIC 2 (density->default | p(x), borderline):")
print(f"  density_pct coef = {coef2:+.3f}  OR={np.exp(coef2):.3f}  p={p2:.2e}")
print(f"  Spearman(density, default) = {rho2:+.3f} (p={prho2:.2e})")
print(f"  V4(Diag2) {'PASS' if V4_D2_PASS else 'FAIL'} (paradox sign = positive)")
print("  (bootstrap CI for this coef is produced in 07_robustness.ipynb)")


# ---------------------------------------------------------------------------
# CELL 6 — V4': the DROPPED ensemble, documented as FAIL.
# Reconstruct the 2-axis ensemble ONLY to show it has no convergent validity,
# which is WHY the ensemble is dropped (R-18). Not used downstream.
# ---------------------------------------------------------------------------
ens = (B["A_Stability"] + B["A_LowDensity"]) / 2.0     # the ex-EngineeredScore
Btmp = B.assign(_ENS=ens)
m_ens = sm.Logit(Btmp["DEFAULT"].values,
                 sm.add_constant(Btmp[["_ENS", "P_VIP"]])).fit(disp=0)
ce, pe = m_ens.params["_ENS"], m_ens.pvalues["_ENS"]
print("V4' — DROPPED 2-axis ensemble (Stability + LowDensity)/2 -> default:")
print(f"  ensemble coef = {ce:+.3f}  p={pe:.2e}  "
      f"-> {'converges' if (pe<0.05 and ce<0) else 'NO convergence (FAIL)'}")
print("  DECISION (R-18): the ensemble has no convergent validity; it is dropped.")
print("  Averaging the two diagnostics dilutes both signals (see V4 vs V4').")


# ---------------------------------------------------------------------------
# CELL 7 — DIAGNOSTIC 1 own validity: Stability does NOT converge with default;
# validate it by GROUP SEPARATION instead (its validity is of a different kind).
#   (a) confirm weak/absent default convergence (honest);
#   (b) show it separates disadvantaged vs advantaged (its actual signal).
# ---------------------------------------------------------------------------
print("Diagnostic 1 (Stability) — own validity (NOT via default):")
# (a) default convergence (expected weak)
m_s = logit(B, ["A_Stability", "P_VIP"])
cs, ps = m_s.params["A_Stability"], m_s.pvalues["A_Stability"]
rho_s, prho_s = stats.spearmanr(B["A_Stability"], B["DEFAULT"])
print(f"  (a) Stability->default coef={cs:+.3f} p={ps:.2e}; "
      f"Spearman={rho_s:+.3f} (p={prho_s:.2e})  "
      f"-> {'weak/none (expected)' if ps>=0.05 else 'unexpected convergence'}")
# (b) group separation (its real validity)
dis = B[B["GROUP"]=="dis_primary"]; adv = B[B["GROUP"]=="advantaged"]
u, pg = stats.mannwhitneyu(dis["A_Stability"], adv["A_Stability"],
                           alternative="two-sided")
gap = dis["A_Stability"].mean() - adv["A_Stability"].mean()
print(f"  (b) group separation dis-adv gap={gap:+.3f} (Mann-Whitney p={pg:.2e})  "
      f"-> {'separates groups (validity)' if pg<0.05 else 'no separation'}")
print("  Interpretation: Diagnostic 1 measures local stability, not default risk.")
print("  Its validity rests on reproducible group separation (Stage 4 + 07 CI).")


# ---------------------------------------------------------------------------
# CELL 8 — Typicality paradox descriptive (borderline), for the paper figure.
# Strong within-Taiwan replication (full approved cohort) is in Stage 5.
# ---------------------------------------------------------------------------
B["dens_q"] = pd.qcut(B["density_pct"], 4, labels=["Q1_low","Q2","Q3","Q4_high"])
para = B.groupby("dens_q").agg(default_rate=("DEFAULT","mean"),
                               n=("DEFAULT","size")).round(4)
print("Typicality paradox — default by density quartile (borderline):")
print(para.to_string())
B["bill_tot"] = B[[f"BILL_AMT{i}" for i in range(1,7)]].sum(axis=1)
B["pay_tot"]  = B[[f"PAY_AMT{i}" for i in range(1,7)]].sum(axis=1)
m_dens = logit(B, ["density_pct", "P_VIP", "LIMIT_BAL", "bill_tot", "pay_tot"])
print(f"Density coef vs default (all raw controls, borderline): "
      f"{m_dens.params['density_pct']:+.3f} (p={m_dens.pvalues['density_pct']:.2e})  "
      f"[+ => dense=risky, paradox]")


# ---------------------------------------------------------------------------
# CELL 9 — Persist per-diagnostic artifacts (NO ensemble column).
# Downstream (04 group tests, 07 bootstrap) read these. A_LowDensity / A_NonFrag
# are kept as appendix columns only.
# ---------------------------------------------------------------------------
keep = ["GROUP","CELL","P_VIP","DEFAULT",
        "A_Stability",        # Diagnostic 1
        "density_pct",        # Diagnostic 2
        "A_LowDensity","A_NonFrag"]   # appendix
B[keep].to_parquet(RESULTS / "stage3_borderline_scored.parquet")
idx[["GROUP","CELL","P_VIP","DEFAULT","is_borderline",
     "A_Stability","density_pct","A_LowDensity","A_NonFrag"]
    ].to_parquet(RESULTS / "stage3_approved_scored.parquet")
print("Saved Stage-3 per-diagnostic artifacts (no ensemble score).")


# ---------------------------------------------------------------------------
# CELL 10 — Stage 3 (A') summary
# ---------------------------------------------------------------------------
print("=" * 70)
print("STAGE 3 (A') — TWO DIAGNOSTICS, ENSEMBLE DROPPED (v12)")
print("=" * 70)
print(f"V2 independence : |corr(Stability, Density)|_raw = {r_stab_dens:.2f}  "
      f"{'PASS' if V2_PASS else 'REVIEW'}")
print(f"V3 not-conf     : Diag1 {'PASS' if V3_flags['Diag1 Stability'] else 'REVIEW'} | "
      f"Diag2 {'PASS' if V3_flags['Diag2 Density'] else 'REVIEW'}")
print(f"V4 Diag2 (density->default) : coef={coef2:+.3f} p={p2:.2e}  "
      f"{'PASS (paradox)' if V4_D2_PASS else 'FAIL'}")
print(f"V4' ensemble (DROPPED)      : coef={ce:+.3f} p={pe:.2e}  "
      f"{'converges' if (pe<0.05 and ce<0) else 'FAIL -> dropped (R-18)'}")
print(f"Diag1 (Stability) validity  : group separation p={pg:.2e} "
      f"(NOT via default; Spearman_default={rho_s:+.3f})")
print("-" * 70)
print("CONCLUSION: no single composite score is valid. Report TWO diagnostics:")
print("  Diag1 Stability  -> validated by reproducible group separation (Stage 4/07)")
print("  Diag2 Typicality -> validated by convergence with default (V4) + Stage 5")
print("=" * 70)

Approved: 11,089  |  borderline: 1,141
Two diagnostics built (NO composite ensemble).
Borderline analysis set: 1141 (defaults: 144)
Diagnostic 1 (Stability) summary:
count    1141.000
mean        0.276
std         0.221
min         0.000
25%         0.095
50%         0.221
75%         0.408
max         0.967
Diagnostic 2 (density_pct) summary:
count    1141.000
mean        0.520
std         0.289
min         0.000
25%         0.291
50%         0.549
75%         0.762
max         0.993
V2 — two-diagnostic RAW correlation (borderline):
  |corr(Stability, Density)| = 0.277  (threshold 0.7)
  V2 PASS (diagnostics are independent)

[reference] RAW correlation incl. excluded NonFragility (appendix):
              Density  Stability  NonFragility
Density         1.000      0.277         0.710
Stability       0.277      1.000         0.303
NonFragility    0.710      0.303         1.000
  Density-NonFragility |r| = 0.710 > 0.7  => NonFragility excluded (R-16)
V3 — not-confidence, per diagnostic